# Job runner

**Run All once and leave it running** (also works with the remote desktop minimized or closed - closing the window only
disconnects the session). Claude puts job scripts into `assignment1_2026/jobs/pending/`; this notebook runs them in the
background: one `gpu_*` job and one `cpu_*` job at a time, each as its own Python process, in name order.
Logs: `jobs/logs/`, heartbeat: `jobs/runner_status.json`, history: `jobs/runner_log.txt`. To stop: create `jobs/STOP`
(or Interrupt this cell - running jobs then keep running on their own).

In [ ]:
# Job runner: picks up assignment1_2026/jobs/pending/*.py and runs each as its own Python process (this kernel's Python,
# working directory submission/) - one "gpu_*" job and one "cpu_*" job at a time, each lane in name order
#   jobs/pending/  -> jobs/running/ -> jobs/done/ (exit code 0) or jobs/failed/
#   jobs/logs/<job>.log      everything the job prints
#   jobs/runner_status.json  heartbeat every 15 s (what runs, what waits, what finished)
#   jobs/STOP                (any content) -> start nothing new, finish the running jobs, end
# a file is only picked up 10 s after its last change (so a half-written file is never started)
import ctypes
import json
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

SUBMISSION_DIR = Path.cwd()                                     # this notebook lives in submission/
JOBS_DIR = SUBMISSION_DIR.parent / "jobs"
FOLDERS = {name: JOBS_DIR / name for name in ("pending", "running", "done", "failed", "logs")}
for folder in FOLDERS.values():
    folder.mkdir(parents=True, exist_ok=True)
STATUS_PATH, STOP_PATH, RUNNER_LOG = JOBS_DIR / "runner_status.json", JOBS_DIR / "STOP", JOBS_DIR / "runner_log.txt"
MIN_FILE_AGE_SECONDS = 10


def now() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def runner_log(line: str):
    with open(RUNNER_LOG, "a", encoding="utf-8") as log_file:
        log_file.write(f"{now()} {line}\n")
    print(f"{now()} {line}", flush=True)


# keeps Windows awake while the runner loops (ES_CONTINUOUS | ES_SYSTEM_REQUIRED), released at the end
def keep_computer_awake(enabled: bool):
    if sys.platform == "win32":
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001 if enabled else 0x80000000)


def lane_of(job_path: Path) -> str:
    return "cpu" if job_path.name.startswith("cpu_") else "gpu"


def start_job(job_path: Path) -> dict:
    running_path = FOLDERS["running"] / job_path.name
    job_path.replace(running_path)
    log_file = open(FOLDERS["logs"] / f"{job_path.stem}.log", "w", encoding="utf-8")
    process = subprocess.Popen([sys.executable, "-u", str(running_path)], cwd=SUBMISSION_DIR, stdout=log_file,
                               stderr=subprocess.STDOUT, env={**__import__("os").environ, "PYTHONUNBUFFERED": "1"})
    runner_log(f"started  {job_path.name} (lane {lane_of(job_path)}, pid {process.pid})")
    return {"path": running_path, "process": process, "log_file": log_file, "started": time.time()}


def finish_job(job: dict) -> str:
    job["log_file"].close()
    exit_code = job["process"].returncode
    target = FOLDERS["done" if exit_code == 0 else "failed"] / job["path"].name
    job["path"].replace(target)
    minutes = (time.time() - job["started"]) / 60
    runner_log(f"{'finished' if exit_code == 0 else 'FAILED  '} {job['path'].name} (exit code {exit_code}, {minutes:.1f} min)")
    return target.name


running = {}                                                     # lane -> job
finished_jobs = []
# a job left in running/ by an earlier runner (e.g. after a restart) is moved back to pending/ to run again
for leftover in FOLDERS["running"].glob("*.py"):
    leftover.replace(FOLDERS["pending"] / leftover.name)
keep_computer_awake(True)
runner_log(f"runner started (Python {sys.executable})")
last_status = 0.0
while True:
    try:
        for lane, job in list(running.items()):
            if job["process"].poll() is not None:
                finished_jobs.append(finish_job(job))
                del running[lane]
        stop_requested = STOP_PATH.exists()
        pending = sorted(FOLDERS["pending"].glob("*.py"))
        if not stop_requested:
            for job_path in pending:
                lane = lane_of(job_path)
                if lane not in running and time.time() - job_path.stat().st_mtime > MIN_FILE_AGE_SECONDS:
                    running[lane] = start_job(job_path)
        if time.time() - last_status > 15:
            STATUS_PATH.write_text(json.dumps({
                "heartbeat": now(), "stop_requested": stop_requested,
                "running": {lane: {"job": job["path"].name, "minutes": round((time.time() - job["started"]) / 60, 1)}
                            for lane, job in running.items()},
                "pending": [path.name for path in sorted(FOLDERS["pending"].glob("*.py"))],
                "finished_this_run": finished_jobs[-30:]}, indent=1), encoding="utf-8")
            last_status = time.time()
        if stop_requested and not running:
            break
    except Exception as error:                                   # never let one bad file stop the runner
        runner_log(f"runner error: {type(error).__name__}: {error}")
    time.sleep(5)
STOP_PATH.unlink(missing_ok=True)
keep_computer_awake(False)
runner_log("runner stopped (STOP file)")